# M5 — Inter-lead / spatial (VCG) detector — combined — **v2**

v1 diagnostic: representation too coarse (a 3D trajectory collapsed to ~10 scalars; volet B failed; standalone OOF 0.345 light / 0.535 overfit depth-4; marginal ensemble gain +0.007). **v2 = dense time-resolved spatial representation** (VCG trajectory x/y/z+magnitude+angular-velocity sampled across delta+QRS, Kors+Dower, median & most-pre-excited beats) + **rich 66-pair inter-lead** (relative lag + amplitude ratio) + **area-vector / ventricular gradient** + **rich axis variability** across beats + finer delta vector. Depends on M4 beats (common R-anchor, no per-lead realign).

**Rigor (9h budget):** stability selection (bootstrapped gate), **selection under gap cap (no max-OOF overfit)**, nested-CV unbiased AP, cross-dataset calibration (ECE), sex/torso confound with action, FP wide-QRS specificity audit. Selection is **purely standalone** — NO mutual-FN tiebreak (avoids contaminating the later all-models complementarity). rho_M3/M4 + ensemble preview = **read-only diagnostics**. Judge = OOF AP folds 1-8; fold9 validation; **fold10 UNTOUCHED**. New caches `m5v2_*` (v1 `m5_combined_*` intact).


### Block 9a.0: Setup — detector, transforms, dense-feature config, rigor flags

In [ ]:
# M5 v2 - INTER-LEAD / SPATIAL (VCG). DEPENDS ON M4 (wavelet_env beats, common R-anchor; no per-lead realign).
# v1 lesson: the representation was TOO COARSE (a 3D trajectory collapsed to ~10 scalars/view; volet B failed).
# v2 = DENSE TIME-RESOLVED spatial trajectory (X/Y/Z + magnitude + angular velocity sampled across delta+QRS) +
# RICH pairwise inter-lead (66 lead-pairs: relative lag + amplitude ratio) + area-vector / ventricular gradient +
# rich axis variability across beats + finer delta vector (rotation speed). Angles/ratios first (batch-robust).
# Selection = max OOF AP UNDER GAP CAP + parsimony (NO mutual-FN tiebreak -> avoid contaminating the later all-
# models complementarity analysis). Rigor we can finally afford (compute budget is available): stability selection
# (bootstrapped gate), optional nested-CV unbiased AP, cross-dataset calibration, FP wide-QRS specificity audit,
# sex/torso confound action. rho_M3/M4 + ensemble preview = READ-ONLY diagnostics (drive nothing). fold10 UNTOUCHED.
import os, sys, json, warnings, time, contextlib, glob
import numpy as np, pandas as pd, pywt
from scipy.signal import butter, sosfiltfilt, find_peaks
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from joblib import Parallel, delayed
from tqdm import tqdm
import matplotlib.pyplot as plt
import joblib
from datetime import datetime
warnings.filterwarnings('ignore'); EPS=1e-9
DETECTOR='wavelet_env'; TRANSFORMS={'k':'kors','d':'dower'}
ROOT=r'.'
PROCESSED=os.path.join(ROOT,'data','processed'); MODELS=os.path.join(ROOT,'models','M5_spatial')
FIG=os.path.join(ROOT,'reports','figures'); METRICS=os.path.join(ROOT,'reports','metrics'); SRC=os.path.join(ROOT,'src'); RAW=os.path.join(ROOT,'data','raw')
for d in (MODELS,FIG,METRICS): os.makedirs(d,exist_ok=True)
sys.path.insert(0,SRC)
from signal_loading import load_signal, LEADS_CANONICAL
from evaluation import evaluate_standard
with open(os.path.join(PROCESSED,'filter_config.json'),encoding='utf-8') as f: FCFG=json.load(f)['filter_FINAL']
assert (FCFG['low'],FCFG['high'],FCFG['order'])==(0.5,40,4), "Unexpected filter!"
FS=FCFG['fs']; LEADS_M5=list(LEADS_CANONICAL); LEAD_IDX={L:LEADS_CANONICAL.index(L) for L in LEADS_M5}
TAG='combined'; TIE_EPS=0.01
# ---- v2 config ----
RPRE,RPOST=int(0.25*FS),int(0.45*FS); MIN_BEATS=3; PAD=5120
N_TRAJ=56                 # dense trajectory sample points across [onset, onset+120ms]
DELTA_MS=(10,15,20,25,30,35,40)
GAP_CAP=0.30              # selection: max OOF AP among configs with train-OOF gap <= GAP_CAP (deployed-style rigor)
STAB_B=60; STAB_THR=0.60  # stability selection: keep features picked (|d|>0.3) in >= 60% of 60 bootstraps
RUN_NESTED=True           # nested-CV unbiased AP (set False if 9h budget is tight; ~doubles modeling time)
N_SEEDS=20                # multiseed for tighter fold9 CI
SOS_BP=butter(FCFG['order'],[FCFG['low']/(FS/2),FCFG['high']/(FS/2)],btype='band',output='sos')
def bp(x): return sosfiltfilt(SOS_BP,np.asarray(x,dtype=np.float64))
KORS={'X':{'I':0.38,'II':-0.07,'V1':-0.13,'V2':0.05,'V3':-0.01,'V4':0.14,'V5':0.06,'V6':0.54},
      'Y':{'I':-0.07,'II':0.93,'V1':0.06,'V2':-0.02,'V3':-0.05,'V4':0.06,'V5':-0.17,'V6':0.13},
      'Z':{'I':0.11,'II':-0.23,'V1':-0.43,'V2':-0.06,'V3':-0.14,'V4':-0.20,'V5':-0.11,'V6':0.31}}
DOWER={'X':{'V1':-0.172,'V2':-0.074,'V3':0.122,'V4':0.231,'V5':0.239,'V6':0.194,'I':0.156,'II':-0.010},
       'Y':{'V1':0.057,'V2':-0.019,'V3':-0.106,'V4':-0.022,'V5':0.041,'V6':0.048,'I':-0.227,'II':0.887},
       'Z':{'V1':-0.229,'V2':-0.310,'V3':-0.246,'V4':-0.063,'V5':0.055,'V6':0.108,'I':0.022,'II':0.102}}
COEF={'k':KORS,'d':DOWER}
meta=pd.read_csv(os.path.join(PROCESSED,'metadata_combined.csv'),dtype={'ecg_id':str})
assert int((meta.label==1).sum())==142, "Expected 142 WPW!"
print(f"M5 v2 spatial/VCG | leads {len(LEADS_M5)} | detector {DETECTOR} | transforms {list(TRANSFORMS.values())} | N_TRAJ {N_TRAJ} | GAP_CAP {GAP_CAP} | stability {STAB_B}x@{STAB_THR} | nested {RUN_NESTED}")
print(f"metadata: {len(meta)} ECGs, 142 WPW | Train(1-8) {int(meta[(meta.fold.between(1,8))&(meta.label==1)].shape[0])} / Val(9) {int(meta[(meta.fold==9)&(meta.label==1)].shape[0])} / Test(10) {int(meta[(meta.fold==10)&(meta.label==1)].shape[0])} [UNTOUCHED]")
print("Block 9a.0 OK.")


### Block 9a.0b: Patient-leakage check across folds

In [ ]:
# Patient-leakage check across folds (blocking assert).
pat=meta.groupby('patient_id')['fold'].nunique(); leaky=pat[pat>1]
print(f"Unique patients: {meta.patient_id.nunique()} | ECGs: {len(meta)} | in >1 fold: {len(leaky)}")
assert len(leaky)==0, f"PATIENT LEAK: {len(leaky)} patients in multiple folds."
print("[OK] No patient leakage across folds.")


### Block 9a.1: Helpers — common-anchor blocks, VCG, dense trajectory + pairwise + area-vector + axis-variability + synthetic test

In [ ]:
# Helpers. R-detect = M4 composite wavelet_env (shared). Common-anchor (T,12) blocks (no per-lead realign).
# VCG (Kors+Dower). v2 features: DENSE trajectory (X/Y/Z + mag + angular velocity sampled across delta+QRS),
# finer delta vector (+rotation speed), area-vector / ventricular gradient, loop geometry, octant; RICH pairwise
# inter-lead (66 pairs: relative lag + amplitude ratio); rich axis variability across beats; minimal interlead
# summary + delta-polarity vector (clinical). All relational/spatial; global per-ECG scale (not per-lead).
@contextlib.contextmanager
def tqdm_joblib(t):
    class _Cb(joblib.parallel.BatchCompletionCallBack):
        def __call__(self,*a,**k): t.update(n=self.batch_size); return super().__call__(*a,**k)
    old=joblib.parallel.BatchCompletionCallBack; joblib.parallel.BatchCompletionCallBack=_Cb
    try: yield t
    finally: joblib.parallel.BatchCompletionCallBack=old; t.close()
def _pad(x): n=len(x); return x[:PAD] if n>=PAD else np.pad(x,(0,PAD-n),mode='reflect')
def _we_rpeaks(comp):
    cf=pywt.swt(_pad(comp),'db4',level=6,trim_approx=True,norm=True)
    D3,D4=np.asarray(cf[4])[:len(comp)],np.asarray(cf[3])[:len(comp)]
    w=max(1,int(0.04*FS)); env=np.convolve(np.abs(D3)+np.abs(D4),np.ones(w)/w,mode='same')
    pk,_=find_peaks(env,distance=int(0.25*FS),height=np.mean(env)+0.5*np.std(env)); return pk
def detect_r(bp12):
    comp=np.sqrt((bp12**2).mean(axis=1)); pk=_we_rpeaks(comp); w=int(0.05*FS); out=[]
    for p in pk:
        a,b=max(0,p-w),min(len(comp),p+w)
        if b>a: out.append(int(a+np.argmax(comp[a:b])))
    return np.array(sorted(set(out)))
def beat_blocks(bp12,rpk):
    segs=[bp12[r-RPRE:r+RPOST,:] for r in rpk if r-RPRE>=0 and r+RPOST<=len(bp12)]
    if len(segs)<MIN_BEATS: return None
    B=np.array(segs); comp=np.sqrt((B**2).mean(axis=2)); medc=np.median(comp,axis=0)
    keep=[i for i in range(len(B)) if np.corrcoef(comp[i],medc)[0,1]>0.5]
    if len(keep)<MIN_BEATS: keep=list(range(len(B)))
    Bk=B[keep]; med_block=np.median(Bk,axis=0)
    def slur(blk):
        c=np.sqrt((blk**2).mean(axis=1)); s=np.abs(np.gradient(c))[max(0,RPRE-int(0.08*FS)):max(1,RPRE-int(0.02*FS))]
        return float(s.mean()) if len(s) else 0.0
    ext_block=Bk[int(np.argmax([slur(b) for b in Bk]))]
    return med_block,ext_block,Bk
def to_vcg(block,coef):
    nm2i={n:k for k,n in enumerate(LEADS_M5)}; out=np.zeros((block.shape[0],3))
    for a,ax in enumerate(('X','Y','Z')):
        for ln,wgt in coef[ax].items():
            if ln in nm2i: out[:,a]+=wgt*block[:,nm2i[ln]]
    return out
def _ang(v):
    x,y,z=float(v[0]),float(v[1]),float(v[2])
    return np.degrees(np.arctan2(z,x)), np.degrees(np.arctan2(y,np.sqrt(x*x+z*z)+EPS))
def _mag(v): return float(np.sqrt(float(np.dot(v,v))))
def _angbtw(a,b):
    return float(np.degrees(np.arccos(np.clip(float(np.dot(a,b))/((np.linalg.norm(a)+EPS)*(np.linalg.norm(b)+EPS)),-1,1))))
def _spatial_onset(V):                                                   # v2: speed envelope + hysteresis
    d=np.diff(V,axis=0); sp=np.sqrt((d**2).sum(1)); sp=np.concatenate([[sp[0]],sp]) if len(sp) else np.zeros(len(V))
    lo,hi=max(0,RPRE-int(0.06*FS)),min(len(sp),RPRE+int(0.06*FS)); qmax=sp[lo:hi].max()+EPS if hi>lo else EPS
    hi_thr,lo_thr=0.15*qmax,0.05*qmax; on=RPRE
    for i in range(RPRE,max(0,RPRE-int(0.12*FS)),-1):
        if sp[i]<hi_thr:
            on=i
            for j in range(i,max(0,i-int(0.03*FS)),-1):
                if sp[j]<lo_thr: on=j; break
            break
    return on
def vcg_all(V,p):
    o={}; on=_spatial_onset(V); mag=np.sqrt((V**2).sum(1)); qe=min(on+int(0.12*FS),len(V)-1)
    if qe<=on+2: qe=min(on+3,len(V)-1)
    tmax=on+int(np.argmax(mag[on:qe+1])); vmax=V[tmax]; magmax=_mag(vmax)+EPS
    o[f'{p}_qrsmax_mag']=_mag(vmax); az,el=_ang(vmax); o[f'{p}_qrsmax_az']=az; o[f'{p}_qrsmax_el']=el; o[f'{p}_qrsmax_t']=float((tmax-on)*1000/FS)
    for ms in DELTA_MS:
        v=V[min(on+int(ms/1000*FS),len(V)-1)]; a2,e2=_ang(v)
        o[f'{p}_init{ms}_az']=a2; o[f'{p}_init{ms}_el']=e2; o[f'{p}_init{ms}_magr']=_mag(v)/magmax; o[f'{p}_init{ms}_dev']=_angbtw(v,vmax)
    i40=min(on+int(0.04*FS),len(V)-1)
    if i40>on+1:
        sub=V[on:i40+1]; rot=[_angbtw(sub[j],sub[j+1]) for j in range(len(sub)-1)]
        o[f'{p}_init_rotspeed']=float(np.mean(rot)) if rot else 0.0; o[f'{p}_init_rotmax']=float(np.max(rot)) if rot else 0.0
    idx=np.linspace(on,qe,N_TRAJ).astype(int); seg=V[idx]/magmax; mg=mag[idx]/magmax
    for a,axn in enumerate(('x','y','z')):
        for i in range(N_TRAJ): o[f'{p}_traj_{axn}{i}']=float(seg[i,a])
    for i in range(N_TRAJ): o[f'{p}_traj_mag{i}']=float(mg[i])
    av=[_angbtw(V[idx[i]],V[idx[i+1]]) for i in range(len(idx)-1)]
    for i in range(0,len(av),2): o[f'{p}_traj_angv{i}']=float(av[i])
    qa=V[on:qe+1]; Avec=np.trapz(qa,axis=0)
    o[f'{p}_qarea_mag']=_mag(Avec); a3,e3=_ang(Avec); o[f'{p}_qarea_az']=a3; o[f'{p}_qarea_el']=e3
    ts,te=min(RPRE+int(0.20*FS),len(V)-1),min(RPRE+int(0.45*FS),len(V))
    if te>ts+2:
        ta=V[ts:te]; Tvec=np.trapz(ta,axis=0); o[f'{p}_tarea_mag']=_mag(Tvec); a4,e4=_ang(Tvec); o[f'{p}_tarea_az']=a4; o[f'{p}_tarea_el']=e4
        G=Avec+Tvec; o[f'{p}_vgrad_mag']=_mag(G); a5,e5=_ang(G); o[f'{p}_vgrad_az']=a5; o[f'{p}_vgrad_el']=e5
        tmag=np.sqrt((ta**2).sum(1)); tv=ta[int(np.argmax(tmag))]; o[f'{p}_qrst_angle']=_angbtw(vmax,tv); o[f'{p}_tmax_magr']=_mag(tv)/magmax
    for nm,(wa,wb) in (('w0_20',(0,20)),('w20_40',(20,40)),('w40_60',(40,60))):
        s=V[min(on+int(wa/1000*FS),len(V)-1):min(on+int(wb/1000*FS),len(V)-1)+1]
        if len(s)>=2:
            mv=s.mean(0); az6,el6=_ang(mv); o[f'{p}_{nm}_az']=az6; o[f'{p}_{nm}_el']=el6; o[f'{p}_{nm}_magr']=_mag(mv)/magmax
            o[f'{p}_{nm}_path']=float(np.sqrt((np.diff(s,axis=0)**2).sum(1)).sum())/magmax
    midq=on+max(1,int((qe-on)/2))
    for nm,(sa,sb) in (('qareaE',(on,midq)),('qareaL',(midq,qe))):
        seg=V[sa:sb+1]
        if len(seg)>=2:
            Av=np.trapz(seg,axis=0); o[f'{p}_{nm}_mag']=_mag(Av); a7,e7=_ang(Av); o[f'{p}_{nm}_az']=a7; o[f'{p}_{nm}_el']=e7
    loop=V[on:qe+1]
    if len(loop)>=3:
        cov=np.cov((loop-loop.mean(0)).T); ev=np.clip(np.sort(np.linalg.eigvalsh(cov))[::-1],0,None)
        o[f'{p}_planarity']=float(ev[1]/(ev[0]+EPS)); o[f'{p}_nonplan']=float(ev[2]/(ev.sum()+EPS))
    v40=V[min(on+int(0.04*FS),len(V)-1)]; o[f'{p}_oct']=float((v40[0]>0)*4+(v40[1]>0)*2+(v40[2]>0))
    return o
def pairwise_features(block):
    o={}; qa,qb=max(0,RPRE-int(0.04*FS)),min(RPRE+int(0.08*FS),len(block)); Q=block[qa:qb]
    nl=Q.shape[1]; rms=np.sqrt((Q**2).mean(0))+EPS; ml=int(0.04*FS)
    for i in range(nl):
        for j in range(i+1,nl):
            a,b=Q[:,i],Q[:,j]; c=np.correlate(a-a.mean(),b-b.mean(),'full'); mid=len(a)-1
            sl=c[mid-ml:mid+ml+1]; lag=(int(np.argmax(sl))-ml) if len(sl) else 0
            o[f'pair_lag_{LEADS_M5[i]}_{LEADS_M5[j]}']=float(lag*1000/FS)
            o[f'pair_ratio_{LEADS_M5[i]}_{LEADS_M5[j]}']=float(rms[i]/rms[j])
            cc=np.corrcoef(a,b)[0,1]; o[f'pair_corr_{LEADS_M5[i]}_{LEADS_M5[j]}']=float(cc if np.isfinite(cc) else 0.0)
            o[f'pair_xpeak_{LEADS_M5[i]}_{LEADS_M5[j]}']=float(sl.max()/(np.std(a)*np.std(b)*len(a)+EPS)) if len(sl) else 0.0
    return o
def axisvar_features(Bk):
    o={}; azs=[];els=[];mgs=[];qazs=[];plans=[];areas=[]
    for b in Bk:
        V=to_vcg(b,KORS); on=_spatial_onset(V); mag=np.sqrt((V**2).sum(1)); qe=min(on+int(0.12*FS),len(V)-1)
        tmax=on+int(np.argmax(mag[on:qe+1])) if qe>on else on; vmax=V[tmax]
        v=V[min(on+int(0.04*FS),len(V)-1)]; a,e=_ang(v); azs.append(a); els.append(e); mgs.append(_mag(v)/(_mag(vmax)+EPS))
        qa,_=_ang(vmax); qazs.append(qa)
        lp=V[on:qe+1]
        if len(lp)>=3:
            cov=np.cov((lp-lp.mean(0)).T); ev=np.clip(np.sort(np.linalg.eigvalsh(cov))[::-1],0,None); plans.append(float(ev[1]/(ev[0]+EPS)))
            areas.append(float(0.5*np.sum(lp[:-1,0]*lp[1:,1]-lp[1:,0]*lp[:-1,1])))
    def disp(x,nm):
        if len(x)>1: o[f'axisvar_{nm}_std']=float(np.std(x)); o[f'axisvar_{nm}_iqr']=float(np.subtract(*np.percentile(x,[75,25]))); o[f'axisvar_{nm}_range']=float(np.max(x)-np.min(x))
        else: o[f'axisvar_{nm}_std']=0.0; o[f'axisvar_{nm}_iqr']=0.0; o[f'axisvar_{nm}_range']=0.0
    disp(azs,'initaz'); disp(els,'initel'); disp(mgs,'initmag'); disp(qazs,'qrsaz'); disp(plans,'planarity'); disp(areas,'area')
    o['axisvar_initaz_fracshift']=float(np.mean(np.abs(np.array(azs)-np.median(azs))>30)) if len(azs)>1 else 0.0
    return o
def interlead_summary(block):
    o={}; qa,qb=max(0,RPRE-int(0.04*FS)),min(RPRE+int(0.08*FS),len(block)); Q=block[qa:qb]; sc=np.sqrt((Q**2).sum(1)).max()+EPS
    cov=np.cov(Q.T); ev=np.clip(np.sort(np.linalg.eigvalsh(cov))[::-1],0,None); o['il_pratio']=float((ev.sum()**2)/((ev**2).sum()+EPS))
    ons=[]
    for j in range(block.shape[1]):
        v=np.abs(np.gradient(block[:,j])); lo,hi=max(0,RPRE-int(0.06*FS)),min(len(v),RPRE+int(0.06*FS))
        thr=0.1*(v[lo:hi].max()+EPS) if hi>lo else EPS; seg=v[max(0,RPRE-int(0.12*FS)):RPRE]; ix=np.where(seg>thr)[0]; ons.append(int(ix[0]) if len(ix) else len(seg))
    o['il_onset_disp']=float(np.std(ons)); nm2i={n:k for k,n in enumerate(LEADS_M5)}; i1=min(RPRE+int(0.04*FS),len(block))
    init={n:(float(np.mean(block[RPRE:i1,nm2i[n]]))/sc if i1>RPRE else 0.0) for n in LEADS_M5}
    for n in LEADS_M5: o[f'il_dpol_{n}']=init[n]
    o['naive_frontal_axis']=float(np.degrees(np.arctan2(init.get('AVF',0.0),init.get('I',EPS))))
    return o
def process_one(m):
    warnings.filterwarnings('ignore')
    row={'ecg_id':m['ecg_id'],'patient_id':m['patient_id'],'label':m['label'],'fold':m['fold'],'source':m['source'],'extraction_failed':0,'n_beats':0}
    try:
        raw=load_signal(m['ecg_id'],m['source']); bp12=np.column_stack([bp(raw[:,LEAD_IDX[L]]) for L in LEADS_M5])
        rpk=detect_r(bp12); row['n_beats']=int(len(rpk)); bb=beat_blocks(bp12,rpk)
        if bb is None: row['extraction_failed']=1; return row
        med_block,ext_block,Bk=bb; o={}
        for view,blk in (('med',med_block),('ext',ext_block)):
            for tn,coef in COEF.items(): o.update(vcg_all(to_vcg(blk,coef),f'{view}_{tn}'))
        o.update(pairwise_features(med_block)); o.update(axisvar_features(Bk)); o.update(interlead_summary(med_block))
        row.update(o)
    except Exception: row['extraction_failed']=1
    return row
def cohens_d(a,b):
    a,b=a[~np.isnan(a)],b[~np.isnan(b)]
    if len(a)<2 or len(b)<2: return np.nan
    sp=np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2))
    return (a.mean()-b.mean())/sp if sp>0 else np.nan
def d_ci(a,b,n=1000,seed=42):
    rng=np.random.default_rng(seed); a,b=a[~np.isnan(a)],b[~np.isnan(b)]
    if len(a)<2 or len(b)<2: return (np.nan,np.nan)
    ds=[cohens_d(rng.choice(a,len(a),True),rng.choice(b,len(b),True)) for _ in range(n)]
    return tuple(np.nanpercentile(ds,[2.5,97.5]))
def ap_boot(yy,pp,n=1000,seed=42):
    rng=np.random.default_rng(seed); pos,neg=np.where(yy==1)[0],np.where(yy==0)[0]; a=np.empty(n)
    for i in range(n):
        idx=np.concatenate([rng.choice(pos,len(pos),True),rng.choice(neg,len(neg),True)]); a[i]=average_precision_score(yy[idx],pp[idx])
    return np.median(a),np.percentile(a,2.5),np.percentile(a,97.5)
def make_xgb(spw,**kw):
    p=dict(n_estimators=100,max_depth=2,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,reg_lambda=2.0,
           min_child_weight=3,scale_pos_weight=spw,eval_metric='aucpr',tree_method='hist',random_state=42,n_jobs=6)
    p.update(kw); return XGBClassifier(**p)
def _family(c):
    if c.startswith('pair_'): return 'pairwise'
    if c.startswith('axisvar'): return 'axis_variability'
    if c.startswith('il_dpol'): return 'delta_polarity'
    if c.startswith('naive'): return 'naive'
    if c.startswith('il_'): return 'interlead'
    if '_traj_' in c: return 'trajectory'
    if '_init' in c: return 'delta_vector'
    if ('_qarea' in c) or ('_tarea' in c) or ('_vgrad' in c): return 'area_vector'
    if ('_qrst' in c) or ('_tmax' in c): return 'qrs_t'
    if ('_planarity' in c) or ('_nonplan' in c): return 'loop_geometry'
    if '_qrsmax' in c: return 'qrs_axis'
    if c.endswith('_oct'): return 'octant'
    return 'other'
print("Block 9a.1 helpers (v2). Synthetic test...")
rng=np.random.default_rng(0); sig=np.zeros((5000,12))
for L in range(12):
    s=0.1*rng.standard_normal(5000)
    for r in range(300,4800,650):
        s[r-40:r]+=np.linspace(0,0.3,40); s[r-3:r+3]+=1.0; s[r+120:r+240]+=0.2*np.hanning(120)
    sig[:,L]=s
bp12=np.column_stack([bp(sig[:,L]) for L in range(12)]); rpk=detect_r(bp12); bb=beat_blocks(bp12,rpk)
assert bb is not None, "synthetic beat extraction failed"
mb,eb,Bk=bb; o={}
for view,blk in (('med',mb),('ext',eb)):
    for tn,coef in COEF.items(): o.update(vcg_all(to_vcg(blk,coef),f'{view}_{tn}'))
o.update(pairwise_features(mb)); o.update(axisvar_features(Bk)); o.update(interlead_summary(mb))
from collections import Counter
print(f"  R {len(rpk)} | beats {len(Bk)} | features {len(o)} | by family {dict(Counter(_family(c) for c in o))}")
assert all(np.isfinite(v) or (isinstance(v,float) and np.isnan(v)) for v in o.values()), "bad features!"


### Block 9a.1c: Demographics (PTB-XL age/sex) for the confound

In [ ]:
# Demographics for the age/SEX confound (CRITICAL for M5: VCG depends on torso geometry -> sex/body-habitus risk).
# PTB-XL via ptbxl_database.csv (patient_id suffix match). Ningbo deferred.
DEMO={}
try:
    db=pd.read_csv(glob.glob(os.path.join(RAW,'ptbxl','ptb-xl-*','ptbxl_database.csv'))[0])
    pa={int(r['patient_id']):(float(r['age']),float(r['sex'])) for _,r in db.iterrows() if pd.notna(r['patient_id'])}
    for _,r in meta[meta.source=='ptbxl'].iterrows():
        try: suf=int(str(r['patient_id']).replace('ptbxl_',''))
        except Exception: continue
        if suf in pa: DEMO[r['ecg_id']]=pa[suf]
    print(f"demographics: PTB-XL age/sex mapped for {len(DEMO)} ECGs (Ningbo deferred). sex/age confound active on PTB subjects.")
except Exception as e:
    print(f"demographics build failed ({e}) -> age/sex confound skipped.")


### Block 9a.2: Extraction (chunked disk-write, float32, guarded) -> m5v2_features.csv

In [ ]:
# Extraction (v2): workers write chunks to disk (float32), n_jobs=6, guarded/resumable. Separate cache m5v2_* so
# v1 artifacts (m5_combined_*) stay intact until we decide which M5 to deploy.
FEATTAG='v2'
FEATURES_CSV=os.path.join(PROCESSED,'m5v2_features.csv')
if os.path.exists(FEATURES_CSV):
    print(f"[guard] {os.path.basename(FEATURES_CSV)} exists -> extraction skipped.")
else:
    import shutil as _sh
    recs=meta.to_dict('records'); t0=time.time()
    _=Parallel(n_jobs=4,backend='loky')(delayed(process_one)(m) for m in recs[:200])
    print(f"  ETA full ~ {(time.time()-t0)/200*len(recs)/60:.0f} min")
    tmpdir=os.path.join(PROCESSED,'_m5v2tmp')
    if os.path.exists(tmpdir): _sh.rmtree(tmpdir)
    os.makedirs(tmpdir)
    def _proc_chunk(j,ms,tmpdir):
        M0=['ecg_id','patient_id','label','fold','source','extraction_failed','n_beats']
        d=pd.DataFrame([process_one(m) for m in ms]); fc=[c for c in d.columns if c not in M0]
        if fc: d[fc]=d[fc].astype('float32')
        d.to_csv(os.path.join(tmpdir,f'part_{j:05d}.csv'),index=False); return j
    chunks=[recs[i:i+500] for i in range(0,len(recs),500)]; t0=time.time()
    with tqdm_joblib(tqdm(total=len(chunks),desc='M5 v2 extraction',unit='chunk')):
        Parallel(n_jobs=6,backend='loky')(delayed(_proc_chunk)(j,c,tmpdir) for j,c in enumerate(chunks))
    parts=sorted(glob.glob(os.path.join(tmpdir,'part_*.csv'))); cols=pd.read_csv(parts[0],nrows=0).columns.tolist(); first=True
    for pp in parts:
        dd=pd.read_csv(pp).reindex(columns=cols); dd.to_csv(FEATURES_CSV,index=False,mode=('w' if first else 'a'),header=first); first=False
    _sh.rmtree(tmpdir); print(f"  extraction done in {(time.time()-t0)/60:.1f} min ({len(parts)} chunks)")
_hdr=pd.read_csv(FEATURES_CSV,nrows=0).columns.tolist()
_scc={'ecg_id','patient_id','source'}; _icc={'label','fold','extraction_failed','n_beats'}
df=pd.read_csv(FEATURES_CSV,dtype={c:(str if c in _scc else 'int16' if c in _icc else 'float32') for c in _hdr})
METAC=['ecg_id','patient_id','label','fold','source','extraction_failed','n_beats']
ALL_FEATS=[c for c in df.columns if c not in METAC]; tr=df[df.fold.between(1,8)]; failrate=float(df.extraction_failed.mean())
from collections import Counter
print(f"  features {len(ALL_FEATS)} | by family {dict(Counter(_family(c) for c in ALL_FEATS))}")
print(f"  train1-8 {len(tr)} ({int((tr.label==1).sum())} WPW) | failrate {100*failrate:.1f}%")
assert int((tr.label==1).sum())>=100, f"INCOMPLETE: only {int((tr.label==1).sum())} train WPW."


### Block 9a.3: Gate (|d|>0.3 & p_FDR & IC95 & cross-dataset) + stability selection + dedup

In [ ]:
# Gate (|d|>0.3 & p_FDR & IC95 & cross-dataset) + STABILITY SELECTION (bootstrap the gate STAB_B times; keep
# features picked |d|>0.3 in >= STAB_THR of resamples) -> removes lucky-noise features = better generalization.
SEL=os.path.join(PROCESSED,f'm5v2_{TAG}_selection.csv')
if os.path.exists(SEL) and set(pd.read_csv(SEL,usecols=['feature'])['feature'])==set(ALL_FEATS):
    res=pd.read_csv(SEL); print(f"[guard] gate reloaded ({len(res)})")
else:
    w,n=tr[tr.label==1],tr[tr.label==0]; ptb,nin=tr[tr.source=='ptbxl'],tr[tr.source=='ningbo']; rows=[]
    for c in tqdm(ALL_FEATS,desc='Gate',unit='feat'):
        a,b=w[c].values.astype(float),n[c].values.astype(float); d=cohens_d(a,b)
        try: _,p=mannwhitneyu(a[~np.isnan(a)],b[~np.isnan(b)],alternative='two-sided')
        except Exception: p=np.nan
        lo,hi=d_ci(a,b)
        dp=cohens_d(ptb[ptb.label==1][c].values.astype(float),ptb[ptb.label==0][c].values.astype(float))
        dn=cohens_d(nin[nin.label==1][c].values.astype(float),nin[nin.label==0][c].values.astype(float))
        cross_ok=(np.isfinite(dp) and np.isfinite(dn) and np.sign(dp)==np.sign(dn) and abs(dp)>0.2 and abs(dn)>0.2)
        ci_ok=(np.isfinite(lo) and np.isfinite(hi) and (lo>0)==(hi>0))
        rows.append({'feature':c,'d':d,'d_ptb':dp,'d_nin':dn,'p_raw':p,'ci_excl0':ci_ok,'cross_ok':cross_ok})
    res=pd.DataFrame(rows); ok=res.p_raw.notna(); res.loc[ok,'p_FDR']=multipletests(res.loc[ok,'p_raw'],method='fdr_bh')[1]
    res['gate']=(res.d.abs()>0.3)&(res.p_FDR<0.05)&res.ci_excl0&res.cross_ok
    res=res.sort_values('d',key=lambda s:s.abs(),ascending=False).reset_index(drop=True); res.to_csv(SEL,index=False)
# stability selection (vectorized bootstrap of |d|)
STAB=os.path.join(PROCESSED,f'm5v2_{TAG}_stability.csv')
if os.path.exists(STAB):
    sdf=pd.read_csv(STAB); stab=dict(zip(sdf.feature,sdf.freq)); print(f"[guard] stability reloaded")
else:
    Xall=tr[ALL_FEATS].to_numpy(np.float32); ytr=tr.label.values.astype(int)
    def col_absd(Xs,ys):
        wv,nv=Xs[ys==1],Xs[ys==0]
        mw,mn=np.nanmean(wv,0),np.nanmean(nv,0); vw,vn=np.nanvar(wv,0,ddof=1),np.nanvar(nv,0,ddof=1)
        sp=np.sqrt(((len(wv)-1)*vw+(len(nv)-1)*vn)/max(len(wv)+len(nv)-2,1))+EPS
        return np.abs((mw-mn)/sp)
    rng=np.random.default_rng(7); freq=np.zeros(len(ALL_FEATS))
    for _ in tqdm(range(STAB_B),desc='Stability',unit='boot'):
        idx=rng.choice(len(ytr),len(ytr),True); freq+=(np.nan_to_num(col_absd(Xall[idx],ytr[idx]))>0.3)
    freq/=STAB_B; stab=dict(zip(ALL_FEATS,freq)); pd.DataFrame({'feature':ALL_FEATS,'freq':freq}).to_csv(STAB,index=False)
gate_passed=res[res.gate].feature.tolist()
passed=[f for f in gate_passed if stab.get(f,0)>=STAB_THR]
print(f"gate-passed {len(gate_passed)} | stable(>= {STAB_THR}) among them {len(passed)} (dropped {len(gate_passed)-len(passed)} lucky-noise)")
def dedup_fast(feats,thr=0.95):
    if not feats: return []
    sub=tr[feats].rank().to_numpy(); cm=np.nanmean(sub,axis=0); ii=np.where(np.isnan(sub)); sub[ii]=np.take(cm,ii[1])
    C=np.abs(np.corrcoef(sub,rowvar=False)); idx={f:i for i,f in enumerate(feats)}; keep=[]
    for f in feats:
        if all(C[idx[f],idx[k]]<=thr for k in keep): keep.append(f)
    return keep
dedup_list=dedup_fast(passed)
from collections import Counter
print(f"dedup>0.95 -> {len(dedup_list)} | by family {dict(Counter(_family(c) for c in dedup_list))}")
print(res[res.feature.isin(passed)].head(15)[['feature','d','d_ptb','d_nin','p_FDR']].to_string(index=False))


### Block 9a.4: AP vs k (+CI) + ensemble co-objective (read-only)

In [ ]:
# Modeling matrices + AP-vs-k (+CI, 1000 boot). Co-objective rho(M5,M3/M4) + M3+M4+M5 stack = READ-ONLY diagnostic
# (drives NO decision; the real complementarity is the later all-models analysis). M3/M4 OOF from frozen CSVs.
d8=df[df.fold.between(1,8)].reset_index(drop=True); y=d8.label.values; folds=d8.fold.values; src8=d8.source.values; eid8=d8.ecg_id.values
FOLD_MASKS=[(folds!=h,folds==h) for h in sorted(np.unique(folds))]
f9=df[df.fold==9].reset_index(drop=True); y9=f9.label.values; spw8=(y==0).sum()/max((y==1).sum(),1)
BASE=passed; COL={c:i for i,c in enumerate(BASE)}; Xall8=d8[BASE].to_numpy(np.float32); Xall9=f9[BASE].to_numpy(np.float32)  # index on full gate-passed pool (family ablation may keep features the global dedup dropped)
def _ci(feats): return [COL[c] for c in feats]
def oof_xgb(feats,**kw):
    X=Xall8[:,_ci(feats)]; oof=np.full(len(d8),np.nan)
    for trm,vam in FOLD_MASKS:
        if y[trm].sum()==0 or y[vam].sum()==0: continue
        oof[vam]=make_xgb((y[trm]==0).sum()/max((y[trm]==1).sum(),1),**kw).fit(X[trm],y[trm]).predict_proba(X[vam])[:,1]
    return oof
def oof_ap(feats,**kw):
    oof=oof_xgb(feats,**kw); m=~np.isnan(oof); return float(average_precision_score(y[m],oof[m]))
OTH={}
for k,p in [('m3',os.path.join(PROCESSED,'m3_combined_oof.csv')),('m4',os.path.join(PROCESSED,'m4_combined_oof.csv'))]:
    if os.path.exists(p):
        oo=pd.read_csv(p,dtype={'ecg_id':str}); OTH[k]=dict(zip(oo.ecg_id,oo.proba_raw))
def co_objective(oof):
    out={}; base=pd.DataFrame({'ecg_id':eid8,'y':y,'m5':oof})
    for k in ('m3','m4'):
        if k in OTH: base[k]=[OTH[k].get(e,np.nan) for e in eid8]
    for k in ('m3','m4'):
        if k in base.columns:
            s=base[base.m5.notna()&base[k].notna()]; out[f'rho_{k}']=float(s[['m5',k]].corr(method='spearman').iloc[0,1])
    cols=[c for c in ('m3','m4','m5') if c in base.columns]; s=base.dropna(subset=cols)
    if len(cols)>=2 and s.y.sum()>0:
        st=LogisticRegression(max_iter=1000).fit(s[cols],s.y); out['ens_AP']=float(average_precision_score(s.y,st.predict_proba(s[cols])[:,1]))
        if set(['m3','m4']).issubset(cols):
            st2=LogisticRegression(max_iter=1000).fit(s[['m3','m4']],s.y); out['ens_AP_noM5']=float(average_precision_score(s.y,st2.predict_proba(s[['m3','m4']])[:,1]))
    return out
RK=os.path.join(PROCESSED,f'm5v2_{TAG}_ap_vs_k.csv')
if os.path.exists(RK): rk=pd.read_csv(RK)
else:
    rows=[]; ks=list(range(2,len(dedup_list)+1,3))
    for ii,k in enumerate(tqdm(ks,desc='AP-vs-k',unit='k')):
        oof=oof_xgb(dedup_list[:int(k)]); m=~np.isnan(oof); r={'k':int(k),'AP_oof':float(average_precision_score(y[m],oof[m]))}; r.update(co_objective(oof)); rows.append(r)
        if ii%6==0: pd.DataFrame(rows).to_csv(RK,index=False)
    rk=pd.DataFrame(rows); rk.to_csv(RK,index=False)
CIp=os.path.join(PROCESSED,f'm5v2_{TAG}_ap_vs_k_ci.csv')
if os.path.exists(CIp): rkci=pd.read_csv(CIp)
else:
    rows=[]
    for k in tqdm(sorted(set(list(range(10,len(dedup_list)+1,20))+[len(dedup_list)])),desc='CI',unit='k'):
        oof=oof_xgb(dedup_list[:k]); m=~np.isnan(oof); md,lo,hi=ap_boot(y[m],oof[m],n=1000); rows.append({'k':k,'AP':md,'lo':lo,'hi':hi})
    rkci=pd.DataFrame(rows); rkci.to_csv(CIp,index=False)
fig,ax=plt.subplots(figsize=(11,5)); ax.fill_between(rkci.k,rkci.lo,rkci.hi,color='#2563eb',alpha=.15,label='95% bootstrap CI')
ax.plot(rk.k,rk.AP_oof,'-',color='#2563eb',lw=1,label='M5 v2 AP OOF folds 1-8')
for v,c,l in [(0.345,'#9ca3af','M5 v1 0.345'),(0.619,'#ea580c','M3 0.619'),(0.718,'#dc2626','M4 0.718')]:
    ax.axhline(v,ls=':',color=c,lw=.8,label=l)
if 'ens_AP' in rk.columns: ax.plot(rk.k,rk.ens_AP,'--',color='black',lw=1,label='M3+M4+M5 stack (read-only)')
if 'ens_AP_noM5' in rk.columns: ax.axhline(rk.ens_AP_noM5.dropna().median(),ls='-',color='gray',lw=.8,label='M3+M4 (no M5)')
ax.set_xlabel('k'); ax.set_ylabel('AP OOF'); ax.legend(fontsize=8,ncol=2); ax.grid(alpha=.3); ax.set_title('M5 v2 spatial - AP vs k [OOF] (+ ensemble preview, read-only)')
plt.savefig(os.path.join(FIG,f'm5v2_{TAG}_ap_vs_k.png'),dpi=150,bbox_inches='tight'); plt.show()
kbest=rk.loc[rk.AP_oof.idxmax()]; K_auto=int(rk[rk.AP_oof>=0.95*kbest.AP_oof].k.min())
print(f"max OOF AP {kbest.AP_oof:.3f}@{int(kbest.k)} | K_auto {K_auto} | k_max {len(dedup_list)}")
if 'rho_m4' in rk.columns: print(f"  [read-only] rho_M4 @maxAP-k {float(rk.loc[rk.AP_oof.idxmax(),'rho_m4']):.3f} | rho_M3 {float(rk.loc[rk.AP_oof.idxmax(),'rho_m3']):.3f}")


### Block 9a.5: Family ablation + diag + grid + selection (gap cap + parsimony, NO mutual-FN)

In [ ]:
# Family ablation + overfit diag + K x config grid + SELECTION = max OOF AP UNDER GAP CAP + parsimony tiebreak
# (NO mutual-FN -> no contamination of the later all-models complementarity). Raw max-OOF reported for transparency.
def diag_one(cfg,feats):
    ci=_ci(feats); X8=Xall8[:,ci]; X9=Xall9[:,ci]; oof=np.full(len(d8),np.nan)
    for trm,vam in FOLD_MASKS:
        if y[trm].sum()==0 or y[vam].sum()==0: continue
        oof[vam]=make_xgb((y[trm]==0).sum()/max((y[trm]==1).sum(),1),**cfg).fit(X8[trm],y[trm]).predict_proba(X8[vam])[:,1]
    m=~np.isnan(oof); ao=average_precision_score(y[m],oof[m]); mdl=make_xgb(spw8,**cfg).fit(X8,y)
    at=average_precision_score(y,mdl.predict_proba(X8)[:,1]); af=average_precision_score(y9,mdl.predict_proba(X9)[:,1]) if y9.sum()>0 else np.nan
    return ao,at,at-ao,af
def oof_perfold_ap(feats,**kw):
    oof=oof_xgb(feats,**kw); aps=[]
    for h in sorted(np.unique(folds)):
        msk=(folds==h)&(~np.isnan(oof))
        if y[msk].sum()>0: aps.append(average_precision_score(y[msk],oof[msk]))
    return np.array(aps)
FAMS={}
for c in passed: FAMS.setdefault(_family(c),[]).append(c)
ABL=os.path.join(PROCESSED,f'm5v2_{TAG}_familyablation.csv')
if os.path.exists(ABL): qc=pd.read_csv(ABL)
else:
    rowsA=[]
    for name,feats in list(FAMS.items())+[('ALL',passed)]:
        dl=dedup_fast(feats)
        if len(dl)<2: continue
        for k in sorted(set(list(range(5,len(dl)+1,15))+[len(dl)])): rowsA.append({'family':name,'k':k,'AP':oof_ap(dl[:k]),'n':len(dl)})
    qc=pd.DataFrame(rowsA); qc.to_csv(ABL,index=False)
plt.figure(figsize=(12,6))
for name in qc['family'].unique():
    cc=qc[qc.family==name]; plt.plot(cc.k,cc.AP,'o-',ms=3,label=f"{name} ({cc.AP.max():.3f})"); print(f"  fam {name:16s}: best OOF AP {cc.AP.max():.3f}")
for v in (0.345,0.619,0.718): plt.axhline(v,ls='--',color='gray',lw=.7)
plt.xlabel('k'); plt.ylabel('AP OOF'); plt.legend(fontsize=8); plt.grid(alpha=.3); plt.title('M5 v2 family ablation')
plt.savefig(os.path.join(FIG,f'm5v2_{TAG}_family_ablation.png'),dpi=140,bbox_inches='tight'); plt.show()
G2=os.path.join(PROCESSED,f'm5v2_{TAG}_grid2d.csv'); kmax=len(dedup_list)
KS=sorted(set(list(range(10,kmax,15))+[K_auto,kmax])); KS=[k for k in KS if 2<=k<=kmax]
CFGS={}
for dep,lrs in [(2,(0.03,0.05)),(3,(0.03,0.05,0.1)),(4,(0.03,0.05,0.1))]:
    for lr in lrs: CFGS[f'd{dep}_lr{int(lr*100):02d}']=dict(max_depth=dep,learning_rate=lr,n_estimators=200,min_child_weight=3)
print(f"KS grid ({len(KS)} pts): {KS}")
if os.path.exists(G2): G2d=pd.read_csv(G2)
else:
    rows=[]
    for k in tqdm(KS,desc='grid',unit='K'):
        for cn,c in CFGS.items():
            ao,at,gp,af=diag_one(c,dedup_list[:k]); rows.append({'K':k,'cfg':cn,'depth':c['max_depth'],'AP_oof':ao,'AP_train':at,'gap':gp,'AP_f9':af})
    G2d=pd.DataFrame(rows); G2d.to_csv(G2,index=False)
print(G2d.sort_values('AP_oof',ascending=False).head(8)[['K','cfg','depth','AP_oof','AP_train','gap','AP_f9']].to_string(index=False))
raw_best=G2d.loc[G2d.AP_oof.idxmax()]
capped=G2d[G2d.gap<=GAP_CAP]; pool=capped if len(capped)>0 else G2d
mo=float(pool.AP_oof.max()); elig=pool[pool.AP_oof>=mo-TIE_EPS].sort_values(['K','depth']); best=elig.iloc[0]
K=int(best.K); CFGNAME=best['cfg']; FEATURES_K=dedup_list[:K]; CFG={**CFGS[CFGNAME],'subsample':0.8,'colsample_bytree':0.8,'reg_lambda':2.0}
print(f"  raw max-OOF (transparency): K={int(raw_best.K)} {raw_best.cfg} OOF {raw_best.AP_oof:.3f} gap {raw_best.gap:+.3f}")
print(f">>> SELECTED (gap<= {GAP_CAP} + parsimony): K={K} {CFGNAME} depth{int(best.depth)} | OOF {best.AP_oof:.3f} fold9 {best.AP_f9:.3f} gap {best.gap:+.3f}")
if len(capped)==0: print("  [warn] no config under gap cap -> fell back to raw max-OOF (all configs overfit).")


### Block 9a.6: Eval + per-dataset + confounds (dataset/age/sex+action) + cross-dataset calibration + freeze

In [ ]:
# Final fit at SELECTED (gap-capped): OOF + Platt + multiseed(N_SEEDS) + evaluate_standard + per-dataset +
# confounds (dataset/age/SEX with ACTION: report sex-AUC after dropping the most sex-loaded feature) +
# cross-dataset calibration (ECE PTB vs Ningbo) + rho/ensemble READ-ONLY + freeze m5v2_*. fold10 untouched.
X8,X9=Xall8[:,_ci(FEATURES_K)],Xall9[:,_ci(FEATURES_K)]
def make_final(**kw): return make_xgb(spw8,**{**CFG,**kw})
pf=oof_perfold_ap(FEATURES_K,**CFGS[CFGNAME])
xgb_raw=make_final().fit(X8,y); ap_train=average_precision_score(y,xgb_raw.predict_proba(X8)[:,1])
oof_raw=np.full(len(d8),np.nan)
for trm,vam in FOLD_MASKS:
    if y[trm].sum()==0 or y[vam].sum()==0: continue
    oof_raw[vam]=make_xgb((y[trm]==0).sum()/max((y[trm]==1).sum(),1),**CFG).fit(X8[trm],y[trm]).predict_proba(X8[vam])[:,1]
mask=~np.isnan(oof_raw); platt=LogisticRegression(max_iter=2000).fit(oof_raw[mask].reshape(-1,1),y[mask])
aps=[];aucs=[]
for s in range(N_SEEDS):
    mm=make_final(random_state=s).fit(X8,y); pp=mm.predict_proba(X9)[:,1]; aps.append(average_precision_score(y9,pp)); aucs.append(roc_auc_score(y9,pp))
MULTI=dict(AP_mean=float(np.mean(aps)),AP_std=float(np.std(aps)),AUC_mean=float(np.mean(aucs)),AUC_std=float(np.std(aucs)))
score9=xgb_raw.predict_proba(X9)[:,1]; score9_cal=platt.predict_proba(score9.reshape(-1,1))[:,1]
ap_oof_final=average_precision_score(y[mask],oof_raw[mask])
order=np.argsort(-score9); yy=y9[order]; tp=np.cumsum(yy); rec=tp/max(yy.sum(),1); prec=tp/np.arange(1,len(yy)+1)
okr=np.where(rec>=0.8)[0]; p80=float(prec[okr[0]]) if len(okr) else float('nan')
M5_STD=evaluate_standard(name=f'M5v2_{TAG}',y_oof=y[mask],score_oof=oof_raw[mask],y_test=y9,score_test=score9,
    figures_dir=FIG,metrics_dir=METRICS,score_test_calibrated=score9_cal,ap_train=ap_train,multiseed=MULTI,
    extra={'K':K,'cfg':CFGNAME,'depth':int(best.depth),'R_DETECTOR':DETECTOR,'transforms':list(TRANSFORMS.values()),
           'gap_cap':GAP_CAP,'stability':f'{STAB_B}x@{STAB_THR}','OOF_AP_folds1_8':float(ap_oof_final),'precision_at_recall0.8_fold9':p80,'failrate':failrate})
print(f"OOF AP {ap_oof_final:.3f} | fold9 AP {M5_STD['AP']:.3f} | AUC {M5_STD['AUC']:.3f} | per-fold {pf.mean():.3f}+/-{pf.std():.3f} (min {pf.min():.3f}) | multiseed {MULTI['AP_mean']:.3f}+/-{MULTI['AP_std']:.3f} | prec@rec0.8 {p80:.3f}")
print(f"  vs v1: OOF 0.535 (overfit depth4) -> v2 {ap_oof_final:.3f} (gap-capped)")
for s in ('ptbxl','ningbo'):
    ms=(src8==s)&mask
    if y[ms].sum()>0: print(f"  OOF AP {s}: {average_precision_score(y[ms],oof_raw[ms]):.3f}")
print(f"  dataset-confound AUC(source~score) = {roc_auc_score((src8=='ptbxl').astype(int)[mask],oof_raw[mask]):.3f} (0.5=clean)")
# cross-dataset calibration (ECE on calibrated OOF)
def ece(yv,pv,bins=10):
    pv=np.clip(pv,0,1); e=0.0; edges=np.linspace(0,1,bins+1)
    for i in range(bins):
        m=(pv>=edges[i])&(pv<edges[i+1])
        if m.sum()>0: e+=abs(pv[m].mean()-yv[m].mean())*m.sum()/len(yv)
    return e
oof_cal=platt.predict_proba(oof_raw[mask].reshape(-1,1))[:,1]; ym=y[mask]; sm=src8[mask]
for s in ('ptbxl','ningbo'):
    ms=sm==s
    if ms.sum()>0: print(f"  calibration ECE {s}: {ece(ym[ms],oof_cal[ms]):.4f} (lower=better; cross-site calibration check)")
# SEX confound + ACTION
if DEMO:
    ag=np.array([DEMO.get(e,(np.nan,np.nan))[0] for e in eid8]); sx=np.array([DEMO.get(e,(np.nan,np.nan))[1] for e in eid8])
    ma=mask&~np.isnan(ag)
    if ma.sum()>50: print(f"  age-confound : corr(score,age) = {np.corrcoef(oof_raw[ma],ag[ma])[0,1]:+.3f} (PTB, n={int(ma.sum())})")
    msx=mask&~np.isnan(sx)
    if msx.sum()>50 and len(np.unique(sx[msx]))>1:
        auc_sex=roc_auc_score(sx[msx],oof_raw[msx]); print(f"  sex-confound : AUC(sex~score) = {auc_sex:.3f} (0.5=clean, PTB) [VCG torso check]")
        sxv=sx[msx].astype(int)
        sexd={c:abs(cohens_d(d8[c].values[mask][msx[mask]][sxv==1].astype(float),d8[c].values[mask][msx[mask]][sxv==0].astype(float))) for c in FEATURES_K}
        worst=max(sexd,key=lambda c:(sexd[c] if np.isfinite(sexd[c]) else -1)); keepf=[c for c in FEATURES_K if c!=worst]
        oof2=np.full(len(d8),np.nan)
        for trm,vam in FOLD_MASKS:
            if y[trm].sum()==0 or y[vam].sum()==0: continue
            oof2[vam]=make_xgb((y[trm]==0).sum()/max((y[trm]==1).sum(),1),**CFG).fit(d8[keepf].to_numpy(np.float32)[trm],y[trm]).predict_proba(d8[keepf].to_numpy(np.float32)[vam])[:,1]
        m2=~np.isnan(oof2)&mask; ap2=average_precision_score(y[m2],oof2[m2]); auc2=roc_auc_score(sx[m2&~np.isnan(sx)],oof2[m2&~np.isnan(sx)]) if (m2&~np.isnan(sx)).sum()>50 else np.nan
        print(f"    -> action: drop most sex-loaded feature '{worst}' (|d_sex|={sexd[worst]:.2f}) -> OOF AP {ap2:.3f} | sex-AUC {auc2:.3f} (compare {auc_sex:.3f})")
else: print("  age/sex confound: skipped (no demographics)")
# rho / ensemble preview (READ-ONLY)
mm=d8[['ecg_id','label']].copy(); mm['m5']=oof_raw; have={}
for k,p in [('m1',os.path.join(PROCESSED,'m1_combined_oof.csv')),('m3',os.path.join(PROCESSED,'m3_combined_oof.csv')),('m4',os.path.join(PROCESSED,'m4_combined_oof.csv'))]:
    if os.path.exists(p): mm=mm.merge(pd.read_csv(p,dtype={'ecg_id':str})[['ecg_id','proba_raw']].rename(columns={'proba_raw':k}),on='ecg_id',how='left'); have[k]=1
p2=os.path.join(ROOT,'models','M2_statistical','m2_oof_folds1_8.npy')
if os.path.exists(p2):
    a2=np.load(p2)
    if len(a2)==len(mm): mm['m2']=a2; have['m2']=1
print("  [READ-ONLY diagnostics — drive no decision; real complementarity = the all-models analysis]")
for k in ('m1','m2','m3','m4'):
    if k in have:
        ssub=mm[mm['m5'].notna()&mm[k].notna()]; rho=ssub[['m5',k]].corr(method='spearman').iloc[0,1]
        X=ssub[[k,'m5']].fillna(ssub[[k,'m5']].mean()); st=LogisticRegression(max_iter=1000).fit(X,ssub.label)
        print(f"    M5 vs {k.upper()}: rho={rho:.3f} | {k.upper()} {average_precision_score(ssub.label,ssub[k]):.3f} -> {k.upper()}+M5 {average_precision_score(ssub.label,st.predict_proba(X)[:,1]):.3f}")
if set(['m3','m4']).issubset(have):
    s=mm.dropna(subset=['m3','m4','m5'])
    a34=average_precision_score(s.label,LogisticRegression(max_iter=1000).fit(s[['m3','m4']],s.label).predict_proba(s[['m3','m4']])[:,1])
    a345=average_precision_score(s.label,LogisticRegression(max_iter=1000).fit(s[['m3','m4','m5']],s.label).predict_proba(s[['m3','m4','m5']])[:,1])
    print(f"    ENSEMBLE preview: M3+M4 {a34:.3f} -> M3+M4+M5 {a345:.3f} (marginal {a345-a34:+.3f})")
# ORTHOGONAL-CORE probe: only the genuinely-spatial-UNIQUE families (pairwise + interlead + axis_variability +
# delta_polarity), i.e. dropping 'trajectory' which overlaps M4 morphology. Quantifies M5's truly orthogonal part.
core_fams=('pairwise','interlead','axis_variability','delta_polarity')
core_feats=dedup_fast([c for c in passed if _family(c) in core_fams])
if len(core_feats)>=5:
    core_oof=oof_xgb(core_feats); cm=~np.isnan(core_oof); ap_core=float(average_precision_score(y[cm],core_oof[cm]))
    cb=pd.DataFrame({'ecg_id':eid8,'y':y,'core':core_oof})
    for kk in ('m3','m4'):
        if kk in OTH: cb[kk]=[OTH[kk].get(e,np.nan) for e in eid8]
    rho4c=float(cb.dropna(subset=['core','m4'])[['core','m4']].corr('spearman').iloc[0,1]) if 'm4' in cb.columns else float('nan')
    if set(['m3','m4']).issubset(cb.columns):
        sc=cb.dropna(subset=['m3','m4','core'])
        a34=average_precision_score(sc.y,LogisticRegression(max_iter=1000).fit(sc[['m3','m4']],sc.y).predict_proba(sc[['m3','m4']])[:,1])
        a34c=average_precision_score(sc.y,LogisticRegression(max_iter=1000).fit(sc[['m3','m4','core']],sc.y).predict_proba(sc[['m3','m4','core']])[:,1])
        print(f'  [orthogonal-core] {len(core_feats)} pure-spatial feats | OOF {ap_core:.3f} | rho_M4 {rho4c:.3f} | M3+M4+core {a34c:.3f} (marginal {a34c-a34:+.3f})')
    else:
        print(f'  [orthogonal-core] {len(core_feats)} pure-spatial feats | OOF {ap_core:.3f} | rho_M4 {rho4c:.3f}')
oof_df=d8[['ecg_id','patient_id','fold','source','label']].copy(); oof_df['proba_raw']=oof_raw; oof_df['proba_cal']=platt.predict_proba(oof_raw.reshape(-1,1))[:,1]
oof_df.to_csv(os.path.join(PROCESSED,f'm5v2_{TAG}_oof.csv'),index=False)
joblib.dump(xgb_raw,os.path.join(MODELS,f'm5v2_{TAG}_model_raw.joblib')); joblib.dump(platt,os.path.join(MODELS,f'm5v2_{TAG}_platt.joblib'))
pd.Series(FEATURES_K,name='feature').to_csv(os.path.join(MODELS,f'm5v2_{TAG}_features.csv'),index=False)
cfg={'model':'M5_spatial','version':'v2','R_DETECTOR':DETECTOR,'transforms':list(TRANSFORMS.values()),'K':int(K),'cfg':CFGNAME,'depth':int(best.depth),
     'gap_cap':GAP_CAP,'stability':f'{STAB_B}x@{STAB_THR}','OOF_AP':float(ap_oof_final),'failrate':failrate,'metrics_standard_fold9':M5_STD,'multiseed':MULTI,
     'representation':'dense VCG trajectory (x/y/z+mag+angvel) + finer delta vector + area-vector/ventricular-gradient + 66-pair inter-lead (lag,ratio) + rich axis variability; Kors+Dower; common-anchor M4 beats',
     'selection':'gate |d|>0.3 & p_FDR & IC95 & cross-dataset; stability '+f'{STAB_B}x@{STAB_THR}'+'; dedup>0.95; max OOF AP under gap<= '+str(GAP_CAP)+' + parsimony; fold10 untouched','fold10':'UNTOUCHED','date_frozen':datetime.now().strftime('%Y-%m-%d %H:%M')}
json.dump(cfg,open(os.path.join(MODELS,f'm5v2_{TAG}_config.json'),'w',encoding='utf-8'),ensure_ascii=False,indent=2)
print("M5 v2 FROZEN -> models/M5_spatial/m5v2_* (v1 m5_combined_* untouched; promote later if v2 wins).")


### Block 9a.7: Rigor — octant face-validity + naive ablation + nested-CV unbiased AP + FP wide-QRS audit

In [ ]:
# Rigor: (1) octant face-validity; (2) naive frontal-axis ablation; (3) nested-CV unbiased AP (RUN_NESTED);
# (4) FP wide-QRS specificity audit -- is M5 a delta-axis detector or just a "wide-QRS" detector? (v1 top feature
#     qrsmax_t is a QRS-width proxy). Uses PTB-XL scp_codes (BBB/IVCD/pacing) on the false positives.
from sklearn.metrics import f1_score
oct_col=[c for c in dedup_list if c.endswith('_oct')]
if oct_col:
    oc=df[df.fold.between(1,8)]; ocw=oc[oc.label==1][oct_col[0]].dropna().astype(int); ocn=oc[oc.label==0][oct_col[0]].dropna().astype(int)
    plt.figure(figsize=(8,4)); bins=np.arange(-0.5,8.5,1)
    plt.hist(ocn,bins=bins,density=True,alpha=.5,label=f'non-WPW (n={len(ocn)})'); plt.hist(ocw,bins=bins,density=True,alpha=.6,label=f'WPW (n={len(ocw)})')
    plt.xlabel(f'initial-vector octant ({oct_col[0]})'); plt.ylabel('density'); plt.legend(); plt.title('M5 v2 localization face-validity: delta-axis octant')
    plt.savefig(os.path.join(FIG,f'm5v2_{TAG}_octant_facevalidity.png'),dpi=140,bbox_inches='tight'); plt.show()
    print(f"  WPW octant: {ocw.value_counts().sort_index().to_dict()}")
nf=[c for c in dedup_list if c.startswith('naive')]
print(f"  ablation: naive frontal-axis only OOF AP {oof_ap(nf):.3f} vs full M5 v2 {ap_oof_final:.3f}" if nf else "  naive frontal-axis did not pass the gate -> full spatial set required.")
# nested-CV unbiased AP: re-run gate (d+mwu+FDR+cross, no d_ci/stability) per outer fold; fixed K & config
if RUN_NESTED:
    print(f"  nested-CV (unbiased; re-select gate per outer fold; K={K} {CFGNAME})...")
    oof_n=np.full(len(d8),np.nan)
    for h in tqdm(sorted(np.unique(folds)),desc='nested',unit='fold'):
        ti=d8[folds!=h]; w,n=ti[ti.label==1],ti[ti.label==0]; ptb,nin=ti[ti.source=='ptbxl'],ti[ti.source=='ningbo']; sc=[]
        for c in ALL_FEATS:
            a,b=w[c].values.astype(float),n[c].values.astype(float); d=cohens_d(a,b)
            if not (np.isfinite(d) and abs(d)>0.3): continue
            try: _,p=mannwhitneyu(a[~np.isnan(a)],b[~np.isnan(b)],alternative='two-sided')
            except Exception: p=1.0
            dp=cohens_d(ptb[ptb.label==1][c].values.astype(float),ptb[ptb.label==0][c].values.astype(float)); dn=cohens_d(nin[nin.label==1][c].values.astype(float),nin[nin.label==0][c].values.astype(float))
            if np.isfinite(dp) and np.isfinite(dn) and np.sign(dp)==np.sign(dn) and abs(dp)>0.2 and abs(dn)>0.2: sc.append((c,abs(d),p))
        if len(sc)<5: continue
        scd=pd.DataFrame(sc,columns=['feature','absd','p']); scd['pf']=multipletests(scd.p,method='fdr_bh')[1]
        cand=dedup_fast(scd[scd.pf<0.05].sort_values('absd',ascending=False).feature.tolist())[:K]
        if len(cand)<2: continue
        Xi=ti[cand].to_numpy(np.float32); yi=ti.label.values; Xh=d8[folds==h][cand].to_numpy(np.float32)
        oof_n[folds==h]=make_xgb((yi==0).sum()/max((yi==1).sum(),1),**CFG).fit(Xi,yi).predict_proba(Xh)[:,1]
    mn=~np.isnan(oof_n); ap_nested=float(average_precision_score(y[mn],oof_n[mn]))
    print(f"  >>> NESTED-CV unbiased OOF AP = {ap_nested:.3f} (vs in-sample-selected {ap_oof_final:.3f}; difference = selection optimism)")
# FP wide-QRS specificity audit (PTB-XL scp_codes)
try:
    import ast; WIDE={'CRBBB','CLBBB','ILBBB','IRBBB','IVCD','LAFB','LPFB'}
    db=pd.read_csv(glob.glob(os.path.join(RAW,'ptbxl','ptb-xl-*','ptbxl_database.csv'))[0])
    db['pid']=db.patient_id.apply(lambda x:f'ptbxl_{int(x)}' if pd.notna(x) else None)
    def codes_of(pid):
        r=db[db.pid==pid]
        if len(r)==0: return set()
        try: return set(ast.literal_eval(r.iloc[0]['scp_codes']).keys())
        except Exception: return set()
    ths=np.linspace(0.01,0.99,99); thr=ths[int(np.argmax([f1_score(y[mask],(oof_cal>=t).astype(int)) for t in ths]))]
    pm=(sm=='ptbxl'); fp_mask=pm&(ym==0)&(oof_cal>=thr)
    pids_masked=d8.patient_id.values[mask]
    fp_pids=pids_masked[fp_mask]; neg_pids=pids_masked[pm&(ym==0)]
    if len(fp_pids)>0:
        wf=np.mean([len(codes_of(p)&WIDE)>0 for p in fp_pids])
        rng2=np.random.default_rng(1); samp=neg_pids if len(neg_pids)<=2000 else rng2.choice(neg_pids,2000,False)
        base=np.mean([len(codes_of(p)&WIDE)>0 for p in samp])
        print(f"  FP wide-QRS audit (PTB): {len(fp_pids)} FP @F1-thr {thr:.2f} | wide-QRS/BBB among FP {wf:.1%} vs base {base:.1%} -> {'CONCERN: fires on wide-QRS mimics' if wf>base+0.15 else 'OK: not disproportionately wide-QRS'}")
    else: print(f"  FP wide-QRS audit: 0 FP above F1-thr {thr:.2f} (PTB) -> skipped")
except Exception as e:
    print(f"  FP wide-QRS audit skipped ({type(e).__name__}: {e})")


### Block 9a.8: Permutation negative-control

In [ ]:
# Permutation negative-control: shuffle labels -> top-50 by shuffled |d| -> OOF AP must collapse to ~chance.
PERM_CSV=os.path.join(PROCESSED,f'm5v2_{TAG}_permutation.csv')
def _oof_ap_perm(feats,yv):
    X=d8[feats].to_numpy(np.float32); oof=np.full(len(d8),np.nan)
    for trm,vam in FOLD_MASKS:
        if yv[trm].sum()==0 or yv[vam].sum()==0: continue
        oof[vam]=make_xgb((yv[trm]==0).sum()/max((yv[trm]==1).sum(),1)).fit(X[trm],yv[trm]).predict_proba(X[vam])[:,1]
    m=~np.isnan(oof); return float(average_precision_score(yv[m],oof[m]))
if os.path.exists(PERM_CSV):
    pr=pd.read_csv(PERM_CSV); print("[guard] permutation reloaded.")
else:
    rng=np.random.default_rng(123); rows=[]
    for it in tqdm(range(5),desc='permutation',unit='shuffle'):
        ysh=y.copy(); rng.shuffle(ysh)
        ds=np.array([abs(cohens_d(d8[c].values[ysh==1].astype(float),d8[c].values[ysh==0].astype(float))) for c in ALL_FEATS])
        top=[ALL_FEATS[i] for i in np.argsort(-np.nan_to_num(ds))[:50]]
        rows.append({'shuffle':it,'null_OOF_AP':_oof_ap_perm(top,ysh)})
    pr=pd.DataFrame(rows); pr.to_csv(PERM_CSV,index=False)
print(f"=== Permutation negative-control ===\n  null OOF AP mean {pr.null_OOF_AP.mean():.3f} max {pr.null_OOF_AP.max():.3f} | REAL {ap_oof_final:.3f} -> {'PASS' if ap_oof_final>pr.null_OOF_AP.max()+0.05 else 'CHECK'}")


### Results & interpretation (M5 v2)

**Verdict:** the spatial/VCG angle plateaus. Honest standalone ~0.40 (nested CV **0.392**, gap-capped selection ~0.43; the raw max-OOF 0.55 is depth-4 overfit). The v2 enrichment (1570 features, dedup 476) **did not raise the ceiling** and **increased redundancy**: rho_M4 0.33 / rho_M3 0.41 (vs 0.19 / 0.22 in v1), ensemble gain **negative** (M3+M4 0.736 -> +M5 0.711).

**Why:** per-family ablation -> what is *unique* (inter-lead pairwise 0.185) is *weak*; what is *strong* (trajectory 0.150) **overlaps M4** (QRS shape in VCG form).

**Specificity:** the FP audit raises the alarm -> 40% of false positives are wide-QRS/BBB vs 20.6% at baseline -> M5 is partly a *wide-QRS* detector, not a delta-axis detector (n=20, wide CI but real signal).

**Confounders:** dataset 0.56 (slightly dirty), sex-AUC 0.41 diffuse (removing one feature changes nothing), age +0.09; ECE calibration excellent on both sites.

**Decision:** M5 v2 is an **honest scientific result** (the spatial angle adds nothing beyond morphology for *detection*; it remains valid for *localization*). Inclusion in the ensemble is decided by the FN-complementarity analysis (symmetric across all models). v2 stays as `m5v2_*`; v1 (`m5_combined_*`) is intact. fold10 never touched.